[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_20_Vector_DBs_in_Production.ipynb)

# Lesson 20 — Vector Databases in Production
**Phase 3 · Production-Grade RAG Infrastructure**

You shipped a working RAG agent in Lesson 7 using **ChromaDB**, an embedded vector store. It was perfect for learning. It is **not** what powers real production RAG systems.

Today you graduate from "vector search demo" to **vector infrastructure**. By the end of this notebook you will:

1. Understand **why** ChromaDB-in-a-notebook breaks at scale (persistence, concurrency, filtering, recall/latency tradeoffs)
2. Internalize **how vector indexes actually work** — flat → IVF → **HNSW** — and the math of recall vs. latency vs. memory
3. Use **three production-class vector stores** hands-on: `sqlite-vec` (the pgvector-style SQL pattern, runs in Colab), **Pinecone** (managed SaaS), and **hnswlib** (the index inside Weaviate/Qdrant/pgvector)
4. Implement **hybrid search** — BM25 (sparse, keyword) + dense embeddings, fused via **Reciprocal Rank Fusion** — and measure why hybrid usually beats dense-only
5. Implement **MMR** (Maximal Marginal Relevance) to diversify results
6. Build a **`VectorStore` abstraction** so your agent can swap backends with one line — the capstone

> 💡 Java analogy: vector databases are to embeddings what RDBMSs are to rows. ChromaDB is HSQLDB in-process. Pinecone is RDS managed Postgres. pgvector is "just add a column to your existing Postgres." You will see all three patterns.

---

## Where this fits

| Lesson | What it gave you |
|---|---|
| L7 — RAG | embeddings, semantic similarity, naive RAG pipeline, ChromaDB in-process |
| **L20 — TODAY** | **production indexes, hybrid search, eval, swappable backends** |
| L21 — Agent Frameworks | CrewAI/AutoGen — orchestration on top of the vector layer |
| L23 — Capstone | AutoResearcher v1.0 with everything wired together |


## 0. Setup

Run this once. We pin reasonably modern versions. The notebook degrades gracefully if optional services (Pinecone) are missing — you only need `ANTHROPIC_API_KEY` and `OPENAI_API_KEY` (for embeddings) to do the full lesson except the Pinecone section.

**Colab Secrets you'll want:**
- `ANTHROPIC_API_KEY` — for the capstone agent
- `OPENAI_API_KEY` — for `text-embedding-3-small` (cheap, fast, 1536-dim)
- `PINECONE_API_KEY` *(optional)* — sign up free at pinecone.io for the managed-DB section


In [ ]:
# Setup: install + import. ~60s on Colab first run.
!pip install -q anthropic openai numpy scikit-learn rank-bm25 hnswlib sqlite-vec pinecone-client tiktoken

import os, time, json, math, sqlite3, struct, random
from typing import List, Dict, Tuple, Optional, Any, Iterable
from dataclasses import dataclass, field

import numpy as np

# ---- API keys (Colab Secrets or os.environ) ----
def load_key(name: str) -> Optional[str]:
    try:
        from google.colab import userdata
        try:
            return userdata.get(name)
        except Exception:
            pass
    except ImportError:
        pass
    return os.environ.get(name)

ANTHROPIC_API_KEY = load_key("ANTHROPIC_API_KEY")
OPENAI_API_KEY    = load_key("OPENAI_API_KEY")
PINECONE_API_KEY  = load_key("PINECONE_API_KEY")  # optional

if OPENAI_API_KEY:    os.environ["OPENAI_API_KEY"]    = OPENAI_API_KEY
if ANTHROPIC_API_KEY: os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

print("OpenAI    :", "OK" if OPENAI_API_KEY    else "missing (need for embeddings)")
print("Anthropic :", "OK" if ANTHROPIC_API_KEY else "missing (need for capstone)")
print("Pinecone  :", "OK" if PINECONE_API_KEY  else "missing (Pinecone section will be skipped)")


---
## 1. Why your ChromaDB demo is not production

ChromaDB-in-a-notebook is a **library**, not a database. Production RAG needs all of these — most of which Chroma-in-process doesn't really give you:

| Concern | What "production" actually requires |
|---|---|
| **Persistence** | Survives process restarts, machine reboots, region failover |
| **Concurrency** | Multiple writers, thousands of QPS readers, no lock contention |
| **Index choice** | Flat (brute force) is O(N·d). At 10M vectors you need an **approximate** index (HNSW, IVF, ScaNN) |
| **Metadata filtering** | `WHERE tenant_id='acme' AND created_at > '2026-01-01'` — fast, indexed, not post-filter |
| **Hybrid search** | Pure semantic search misses exact-match keywords (model numbers, IDs, names). Production blends BM25 + dense |
| **Recall measurement** | "Did we retrieve the *right* chunks?" — you need a labeled set and a metric |
| **Reindexing** | Embedding model upgrade = rebuild every vector. Plan for it on day 1 |
| **Cost** | Embeddings, storage, query cost — observable per-tenant |

The three production patterns:

1. **pgvector** — add a `vector` column to your existing Postgres. Transactional, joins with your row data, ops team already knows Postgres. **Best for most teams.**
2. **Managed SaaS** — Pinecone, Weaviate Cloud, Qdrant Cloud. Pay someone else to run it. Best for fast startups.
3. **Dedicated vector DB self-hosted** — Weaviate, Qdrant, Milvus. You run them. Best at extreme scale.

We will touch all three patterns. In Colab we use `sqlite-vec` as a stand-in for pgvector (same SQL pattern, no server to install), and Pinecone for the managed pattern. `hnswlib` shows you the *index itself*.


---
## 2. The index, from the inside — Flat → IVF → HNSW

Every production vector DB is, at its core, an **approximate nearest neighbor (ANN) index** wrapped in a database. Three families dominate.

### 2.1 Flat (brute force)
Compute distance from the query vector to **every** vector in the corpus. O(N·d).
- Recall: **100%** (exact).
- Latency: linear in N. Fine to ~100K vectors. Painful at 10M+.
- Use when: small corpus, or as the ground-truth oracle for measuring recall of approximate indexes.

### 2.2 IVF (Inverted File)
1. K-means cluster all vectors into `nlist` cells.
2. Query → find `nprobe` nearest cells → brute-force inside those cells only.
- Recall: tunable via `nprobe`. Higher = slower but more accurate.
- Memory: low (just cluster centroids + the vectors).
- Used by FAISS-IVF, Milvus, ScaNN's coarse stage.

### 2.3 HNSW (Hierarchical Navigable Small World)
The dominant index in 2025. Used by Weaviate, Qdrant, pgvector (default), Pinecone (under the hood), Milvus.
- Builds a **multi-layer graph**. Top layer = sparse highway, bottom layer = dense local. You enter at the top, greedily descend toward the query.
- Recall: **>95% typical** with default params.
- Latency: ~ms even at 10M+ vectors.
- Memory: **high** — graph is ~1.5-2× the vector size on top of the vectors themselves.

**Key HNSW parameters you must understand:**
- `M` (default 16) — max connections per node per layer. ↑ → better recall, more memory.
- `ef_construction` (default 200) — neighbors considered when **building**. ↑ → better graph, slower indexing.
- `ef` (search-time, default 50) — neighbors considered when **querying**. ↑ → better recall, slower query.

Let's actually build one and measure recall.


In [ ]:
# Build a synthetic corpus and benchmark FLAT vs HNSW.
import hnswlib, numpy as np, time

rng = np.random.default_rng(42)
D = 128          # dimension
N = 50_000       # corpus size
Q = 200          # queries
K = 10           # top-K

corpus  = rng.normal(size=(N, D)).astype(np.float32)
corpus /= np.linalg.norm(corpus, axis=1, keepdims=True)   # unit-norm → cosine = dot

queries  = rng.normal(size=(Q, D)).astype(np.float32)
queries /= np.linalg.norm(queries, axis=1, keepdims=True)

# ---- Ground truth via FLAT (brute force) ----
t0 = time.time()
sims  = queries @ corpus.T                                # (Q, N)
gt    = np.argpartition(-sims, K, axis=1)[:, :K]
gt    = np.take_along_axis(gt, np.argsort(-np.take_along_axis(sims, gt, 1), axis=1), 1)
flat_ms = (time.time() - t0) * 1000 / Q
print(f"FLAT  : {flat_ms:6.2f} ms/query, recall@{K} = 1.000 (by definition)")

# ---- HNSW ----
index = hnswlib.Index(space='cosine', dim=D)
index.init_index(max_elements=N, M=16, ef_construction=200)
t0 = time.time()
index.add_items(corpus, np.arange(N))
build_s = time.time() - t0
print(f"HNSW build: {build_s:.1f}s for N={N}")

def hnsw_recall(ef):
    index.set_ef(ef)
    t0 = time.time()
    labels, _ = index.knn_query(queries, k=K)
    ms = (time.time() - t0) * 1000 / Q
    recall = np.mean([len(set(l) & set(g)) / K for l, g in zip(labels, gt)])
    return ms, recall

print("\nHNSW ef-vs-recall tradeoff:")
print(f"{'ef':>5}  {'ms/query':>10}  {'recall@10':>10}")
for ef in [10, 30, 50, 100, 200, 400]:
    ms, r = hnsw_recall(ef)
    print(f"{ef:>5}  {ms:>10.2f}  {r:>10.3f}")

# 💡 EXPERIMENT: rebuild with M=8 vs M=32 and see how the recall curve shifts.
# 💡 EXPERIMENT: increase N to 200_000 — watch FLAT get slow, HNSW stay flat.


**What you should see**: as `ef` goes up, recall climbs toward 1.0, latency climbs roughly linearly. That's the dial every production vector DB exposes. Picking the right `ef` is a per-application decision: a customer-support bot may be fine at recall=0.9 / 2ms; a legal-discovery system wants recall=0.99 / 20ms.

**Java mental model**: HNSW is to vectors what a B-tree is to integers — an index structure with knobs that trade space and build time for query speed.


---
## 3. Pattern A — Postgres-style vectors with `sqlite-vec`

In production, pgvector is the default choice for most teams. The pattern:

```sql
CREATE EXTENSION vector;
CREATE TABLE docs (id BIGSERIAL PRIMARY KEY, tenant_id TEXT, body TEXT, embedding vector(1536));
CREATE INDEX ON docs USING hnsw (embedding vector_cosine_ops);
SELECT id, body FROM docs
 WHERE tenant_id = 'acme'
 ORDER BY embedding <=> $1   -- cosine distance
 LIMIT 10;
```

You cannot easily run Postgres in Colab — but `sqlite-vec` is the **same pattern** in SQLite: a vector column, a SQL `MATCH` query, and metadata filtering via plain `WHERE` clauses. The mental model transfers 1:1 to pgvector.

We'll embed a small real corpus, store it with metadata, and run filtered semantic queries.


In [ ]:
# Build a tiny real corpus from the AI/ML domain (so the embeddings are meaningful).
DOCS = [
    {"id":  1, "tenant": "acme",   "year": 2023, "topic": "transformers",  "body": "The Transformer architecture replaced RNNs by using self-attention to model long-range dependencies in sequences."},
    {"id":  2, "tenant": "acme",   "year": 2023, "topic": "transformers",  "body": "Multi-head attention lets each head specialize in a different relational pattern, improving representational power."},
    {"id":  3, "tenant": "acme",   "year": 2024, "topic": "rag",           "body": "Retrieval-Augmented Generation grounds LLM answers in retrieved documents, reducing hallucination and enabling fresh knowledge."},
    {"id":  4, "tenant": "acme",   "year": 2024, "topic": "rag",           "body": "Chunking strategy strongly influences RAG recall: overlap and semantic boundaries matter more than chunk size alone."},
    {"id":  5, "tenant": "acme",   "year": 2025, "topic": "agents",        "body": "An LLM agent is a loop: the model proposes a tool call, the runtime executes it, the result is fed back, and the loop continues until a final answer."},
    {"id":  6, "tenant": "acme",   "year": 2025, "topic": "agents",        "body": "Tool use turns an LLM into a planner — the model decides which API to call rather than answering from its weights alone."},
    {"id":  7, "tenant": "globex", "year": 2024, "topic": "vector-db",     "body": "HNSW is a graph-based ANN index that achieves sub-millisecond queries at million-vector scale with >95% recall."},
    {"id":  8, "tenant": "globex", "year": 2024, "topic": "vector-db",     "body": "pgvector stores embeddings as a native Postgres column type, enabling vector search inside transactional SQL queries."},
    {"id":  9, "tenant": "globex", "year": 2025, "topic": "vector-db",     "body": "Hybrid search combines BM25 keyword scores with dense vector similarity, fused via reciprocal rank fusion, and consistently outperforms either alone."},
    {"id": 10, "tenant": "globex", "year": 2025, "topic": "evals",         "body": "Faithfulness in RAG asks whether each claim in the answer is supported by the retrieved passages — measured by claim extraction plus verification."},
    {"id": 11, "tenant": "globex", "year": 2023, "topic": "fine-tuning",   "body": "LoRA freezes the base model and trains low-rank adapter matrices, cutting GPU memory by 10x compared to full fine-tuning."},
    {"id": 12, "tenant": "acme",   "year": 2025, "topic": "agents",        "body": "ReAct interleaves reasoning traces with tool-use actions, letting the model self-correct mid-trajectory."},
]
print(f"Corpus size: {len(DOCS)} documents")


In [ ]:
# Embed the corpus with OpenAI text-embedding-3-small (1536-dim).
from openai import OpenAI
oai = OpenAI()
EMBED_MODEL = "text-embedding-3-small"
EMBED_DIM   = 1536

def embed(texts: List[str]) -> np.ndarray:
    resp = oai.embeddings.create(model=EMBED_MODEL, input=texts)
    vecs = np.array([d.embedding for d in resp.data], dtype=np.float32)
    vecs /= np.linalg.norm(vecs, axis=1, keepdims=True)   # unit-norm for cosine
    return vecs

texts = [d["body"] for d in DOCS]
embeddings = embed(texts)
print("Embeddings shape:", embeddings.shape)


In [ ]:
# Now store them in sqlite-vec — same SQL pattern as pgvector.
import sqlite3, sqlite_vec

con = sqlite3.connect(":memory:")
con.enable_load_extension(True)
sqlite_vec.load(con)
con.enable_load_extension(False)

# Two tables: metadata in normal SQL, vectors in a vec0 virtual table keyed by rowid.
con.executescript(f'''
    CREATE TABLE docs(
        id      INTEGER PRIMARY KEY,
        tenant  TEXT,
        year    INTEGER,
        topic   TEXT,
        body    TEXT
    );
    CREATE VIRTUAL TABLE doc_vecs USING vec0(
        embedding float[{EMBED_DIM}]
    );
''')

# Insert
for d, vec in zip(DOCS, embeddings):
    con.execute(
        "INSERT INTO docs(id, tenant, year, topic, body) VALUES(?, ?, ?, ?, ?)",
        (d["id"], d["tenant"], d["year"], d["topic"], d["body"]),
    )
    con.execute(
        "INSERT INTO doc_vecs(rowid, embedding) VALUES(?, ?)",
        (d["id"], vec.tobytes()),
    )
con.commit()
print("Inserted", con.execute("SELECT count(*) FROM docs").fetchone()[0], "docs")


In [ ]:
# Filtered semantic search — pgvector-style.
def vec_search(query: str, k: int = 3, tenant: Optional[str] = None,
               topic: Optional[str] = None, year_min: Optional[int] = None) -> List[Dict]:
    q_vec = embed([query])[0]
    # Step 1: ANN candidate set from the vector index
    rows = con.execute('''
        SELECT rowid, distance
          FROM doc_vecs
         WHERE embedding MATCH ?
         ORDER BY distance
         LIMIT ?
    ''', (q_vec.tobytes(), k * 5)).fetchall()                    # over-fetch, filter later

    # Step 2: join with metadata + filter
    out = []
    for rowid, dist in rows:
        meta = con.execute(
            "SELECT id, tenant, year, topic, body FROM docs WHERE id = ?", (rowid,)
        ).fetchone()
        if meta is None:
            continue
        _id, t, y, tp, body = meta
        if tenant   is not None and t  != tenant:   continue
        if topic    is not None and tp != topic:    continue
        if year_min is not None and y  <  year_min: continue
        out.append({"id": _id, "tenant": t, "year": y, "topic": tp,
                    "body": body, "distance": float(dist)})
        if len(out) >= k:
            break
    return out

print("Q: 'how do agents use tools?'  filter tenant=acme, year>=2024")
for r in vec_search("how do agents use tools?", k=3, tenant="acme", year_min=2024):
    print(f"  [{r['distance']:.3f}] ({r['topic']}/{r['year']}) {r['body'][:90]}...")

print("\nQ: 'graph-based ANN index for vectors'  no filter")
for r in vec_search("graph-based ANN index for vectors", k=3):
    print(f"  [{r['distance']:.3f}] ({r['tenant']}/{r['topic']}) {r['body'][:90]}...")

# 💡 EXPERIMENT: try post-filtering vs pre-filtering. With huge corpora,
# pre-filter (WHERE tenant=...) before the vector index, otherwise you may
# fetch K nearest, then filter them out and end up with <K results.


**Two filtering patterns to know**:

- **Pre-filter** — narrow with metadata first (`WHERE tenant_id='acme'`), then ANN. pgvector + Postgres planner does this when selective. Safe.
- **Post-filter** — ANN first, then drop non-matching rows. Cheap. **Dangerous**: if the filter is selective (e.g., 0.1% of corpus), you may need to over-fetch by 1000× to get K survivors.

Real systems use **pre-filter with a partial HNSW index per tenant**, or per-tenant **namespaces** (Pinecone's model).


---
## 4. Pattern B — Managed SaaS with Pinecone

When you don't want to run a database, Pinecone is the canonical choice. The mental model:

- **Index** = the database (one vector dimension, one metric).
- **Namespace** = soft partition inside an index, typically per-tenant. Free.
- **Serverless** = pay per query + storage, scales to zero. Free tier exists.

We'll create an index, upsert with metadata, query with a filter, and clean up. The code is **identical** in shape to pgvector — that's the point: once you've internalized the vector-DB API surface, the vendor is interchangeable.

Skip this cell if you don't have a Pinecone API key — the rest of the notebook does not depend on it.


In [ ]:
# Optional Pinecone section. Set PINECONE_API_KEY in Colab Secrets to run.
if not PINECONE_API_KEY:
    print("Skipping Pinecone — no API key. Sign up free at pinecone.io and re-run.")
else:
    from pinecone import Pinecone, ServerlessSpec
    pc = Pinecone(api_key=PINECONE_API_KEY)
    INDEX_NAME = "learn-ai-lesson20"

    # Create the index if missing. cosine + serverless free tier (us-east-1 AWS).
    existing = [i.name for i in pc.list_indexes()]
    if INDEX_NAME not in existing:
        pc.create_index(
            name=INDEX_NAME,
            dimension=EMBED_DIM,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        )
        # wait for ready
        while not pc.describe_index(INDEX_NAME).status["ready"]:
            time.sleep(1)
        print("Created Pinecone index:", INDEX_NAME)
    else:
        print("Using existing Pinecone index:", INDEX_NAME)

    idx = pc.Index(INDEX_NAME)

    # Upsert with metadata. Each tenant gets its own namespace — Pinecone's
    # canonical multi-tenant pattern. Cheaper than one index per tenant.
    by_tenant: Dict[str, List] = {}
    for d, vec in zip(DOCS, embeddings):
        by_tenant.setdefault(d["tenant"], []).append({
            "id": str(d["id"]),
            "values": vec.tolist(),
            "metadata": {"topic": d["topic"], "year": d["year"], "body": d["body"]},
        })
    for ns, vecs in by_tenant.items():
        idx.upsert(vectors=vecs, namespace=ns)
    time.sleep(2)  # eventual consistency
    print("Stats:", idx.describe_index_stats())

    # Query within tenant=acme namespace, filtered to year>=2024
    q = embed(["how do agents use tools?"])[0].tolist()
    res = idx.query(
        namespace="acme",
        vector=q,
        top_k=3,
        filter={"year": {"$gte": 2024}},
        include_metadata=True,
    )
    print("\nQ: 'how do agents use tools?'  namespace=acme, year>=2024")
    for m in res["matches"]:
        print(f"  [{m['score']:.3f}] ({m['metadata']['topic']}/{m['metadata']['year']}) "
              f"{m['metadata']['body'][:90]}...")

    # 💡 EXPERIMENT: try namespace='globex' with the same query — different answers.
    # 💡 EXPERIMENT: pc.delete_index(INDEX_NAME) when done to keep your free tier clean.


Notice how **identical** Pinecone's `query()` is to the sqlite-vec SQL: vector + filter + top_k. The vendor changed; the abstraction did not. That's exactly what we'll exploit in the capstone.

**Production gotchas for managed vector DBs:**
- **Eventual consistency** — upserts are not instantly queryable. The `time.sleep(2)` above is real.
- **Per-namespace cost** — many namespaces is fine; many *indexes* gets expensive.
- **Filter expressivity** — Pinecone's filter is JSON-style (`$gte`, `$in`), not SQL. Test your filter syntax.


---
## 5. Hybrid search — dense + BM25 + RRF

**Problem with dense-only search**: embeddings are semantically smooth but **lose exact matches**. Search for `"GPT-4o"` and a dense model may return "the latest OpenAI model" — semantically close but maybe not the exact thing the user typed. For names, model numbers, error codes, SKUs, BM25 (TF-IDF-ish keyword search) crushes dense.

**Production answer: hybrid.** Run both, fuse the rankings.

### Reciprocal Rank Fusion (RRF)
A beautifully simple algorithm:

$$\text{score}_{\text{RRF}}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)}$$

with `k` typically = 60. **No score normalization needed** — only ranks. This is what Weaviate, Elasticsearch, and Vespa use under the hood.

We'll implement it from scratch so you can read its 8 lines anywhere.


In [ ]:
from rank_bm25 import BM25Okapi
import re

def tokenize(text: str) -> List[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

# Build a BM25 index over the same corpus
tokenized_corpus = [tokenize(d["body"]) for d in DOCS]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query: str, k: int = 10) -> List[Tuple[int, float]]:
    scores = bm25.get_scores(tokenize(query))
    order  = np.argsort(-scores)[:k]
    return [(DOCS[i]["id"], float(scores[i])) for i in order if scores[i] > 0]

def dense_search(query: str, k: int = 10) -> List[Tuple[int, float]]:
    q_vec = embed([query])[0]
    sims  = embeddings @ q_vec
    order = np.argsort(-sims)[:k]
    return [(DOCS[i]["id"], float(sims[i])) for i in order]

def rrf_fuse(rankings: List[List[Tuple[int, float]]], k: int = 60, top_k: int = 5) -> List[Tuple[int, float]]:
    """Reciprocal Rank Fusion. Each ranking is [(doc_id, score), ...] best-first."""
    fused: Dict[int, float] = {}
    for ranking in rankings:
        for rank, (doc_id, _score) in enumerate(ranking):
            fused[doc_id] = fused.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(fused.items(), key=lambda x: -x[1])[:top_k]

def hybrid_search(query: str, top_k: int = 5) -> List[Dict]:
    dense  = dense_search(query, k=10)
    sparse = bm25_search(query, k=10)
    fused  = rrf_fuse([dense, sparse], top_k=top_k)
    id_to_doc = {d["id"]: d for d in DOCS}
    return [{"id": did, "rrf_score": s, "body": id_to_doc[did]["body"]} for did, s in fused]

# ---- Compare on a query that benefits from hybrid ----
QUERY = "LoRA adapter low-rank fine-tuning"

print("DENSE only:")
for did, s in dense_search(QUERY, k=5):
    print(f"  [{s:.3f}] doc#{did}  {next(d['body'] for d in DOCS if d['id']==did)[:80]}")

print("\nBM25 only:")
for did, s in bm25_search(QUERY, k=5):
    print(f"  [{s:.3f}] doc#{did}  {next(d['body'] for d in DOCS if d['id']==did)[:80]}")

print("\nHYBRID (RRF):")
for r in hybrid_search(QUERY, top_k=5):
    print(f"  [{r['rrf_score']:.4f}] doc#{r['id']}  {r['body'][:80]}")

# 💡 EXPERIMENT: try query='attention heads transformers' — both should agree.
# 💡 EXPERIMENT: try query='HNSW' — pure keyword match; BM25 dominates.
# 💡 EXPERIMENT: try query='loops of reasoning' — semantic; dense dominates.


When you go to production, every mature vector DB exposes hybrid as a first-class feature:
- **Weaviate**: `nearVector` + `bm25` + `hybrid` operator, native RRF.
- **Pinecone**: sparse-dense indexes (you upsert both dense and sparse vectors).
- **pgvector**: dense + Postgres full-text (`tsvector`) + custom RRF in SQL.

Knowing the algorithm (it's 8 lines) means you can implement it anywhere, including on top of your own dense+BM25 combo.


---
## 6. Diversification with MMR

A second production problem: **top-K is often near-duplicates**. If you have 100 docs about "RAG chunking" and ask for top-5, all 5 might say the same thing. The LLM gets repetitive context.

**MMR (Maximal Marginal Relevance)** picks each next result to maximize:

$$\lambda \cdot \text{sim}(d, q) - (1 - \lambda) \cdot \max_{d' \in \text{selected}} \text{sim}(d, d')$$

`λ` is the relevance/diversity dial. `λ=1` → pure relevance (original ranking). `λ=0` → maximum diversity. Production sweet spot is usually `λ=0.5–0.7`.


In [ ]:
def mmr(query_vec: np.ndarray, candidate_vecs: np.ndarray, candidate_ids: List[int],
        k: int = 5, lam: float = 0.5) -> List[int]:
    """Maximal Marginal Relevance over a candidate set."""
    sims_to_q = candidate_vecs @ query_vec                       # (C,)
    selected_idx: List[int] = []
    remaining = set(range(len(candidate_ids)))
    while len(selected_idx) < min(k, len(candidate_ids)):
        best, best_score = None, -1e9
        for i in remaining:
            if selected_idx:
                sims_to_sel = candidate_vecs[selected_idx] @ candidate_vecs[i]
                redundancy  = float(np.max(sims_to_sel))
            else:
                redundancy = 0.0
            score = lam * float(sims_to_q[i]) - (1 - lam) * redundancy
            if score > best_score:
                best, best_score = i, score
        selected_idx.append(best)
        remaining.remove(best)
    return [candidate_ids[i] for i in selected_idx]

# Demo: pretend "transformers" docs (1, 2) are near-duplicates from a user's POV.
# Without MMR, dense top-2 might return both. With MMR(lam=0.3), we should get diversity.
q_vec       = embed(["explain modern AI architectures"])[0]
cand_ids    = [d["id"] for d in DOCS]
cand_vecs   = embeddings

print(f"Pure dense top-5 (no MMR):")
for did, s in dense_search("explain modern AI architectures", k=5):
    print(f"  doc#{did}  {next(d['body'] for d in DOCS if d['id']==did)[:80]}")

for lam in [0.9, 0.5, 0.2]:
    picks = mmr(q_vec, cand_vecs, cand_ids, k=5, lam=lam)
    print(f"\nMMR lam={lam}  (lam=1 → pure relevance, lam=0 → pure diversity):")
    for did in picks:
        print(f"  doc#{did}  {next(d['body'] for d in DOCS if d['id']==did)[:80]}")

# 💡 EXPERIMENT: at lam=0.2 you should see a 'transformers' doc AND an 'agents'
# doc AND a 'rag' doc — much broader coverage than lam=0.9.


When to use MMR: when your retrieved context will be fed into an LLM and you'd rather have 5 *different* perspectives than 5 phrasings of the same one. ChromaDB, LangChain, and LlamaIndex all expose MMR as a one-flag option (`search_type="mmr"`); now you know what it actually does.


---
## 7. Measuring retrieval quality

You cannot optimize what you don't measure. The vector-search metrics you'll hear about in production:

| Metric | What it asks | When you care |
|---|---|---|
| **Recall@K** | Of the truly relevant docs, what fraction are in top-K? | Always |
| **Precision@K** | Of the top-K, what fraction are relevant? | Less common in RAG (LLM filters) |
| **MRR** (Mean Reciprocal Rank) | 1/rank of the first relevant doc, averaged | When **one** right answer matters |
| **NDCG@K** | Graded relevance, position-discounted | When relevance has degrees, not binary |

For a RAG system, **Recall@K** is usually the most actionable: "Of the chunks the LLM *needs* to answer correctly, how often do we put them in the context window?"

We'll build a tiny labeled set and compare DENSE, BM25, and HYBRID.


In [ ]:
# Hand-labeled relevance: for each query, the doc IDs we *know* are relevant.
LABELED = [
    ("how do agents use tools?",                {5, 6, 12}),
    ("low-rank fine-tuning",                    {11}),
    ("retrieval augmented generation grounds answers", {3, 4}),
    ("ANN graph index",                         {7}),
    ("hybrid search combining BM25 and vectors",{9}),
    ("multi-head attention",                    {1, 2}),
]

def recall_at_k(retrieved_ids: List[int], relevant: set, k: int) -> float:
    if not relevant:
        return 0.0
    top_k = set(retrieved_ids[:k])
    return len(top_k & relevant) / len(relevant)

def mrr(retrieved_ids: List[int], relevant: set) -> float:
    for rank, did in enumerate(retrieved_ids, start=1):
        if did in relevant:
            return 1.0 / rank
    return 0.0

K = 3
results = {"dense": [], "bm25": [], "hybrid": []}
mrrs    = {"dense": [], "bm25": [], "hybrid": []}

for query, relevant in LABELED:
    d_ids = [did for did, _ in dense_search(query, k=10)]
    b_ids = [did for did, _ in bm25_search(query, k=10)]
    h_ids = [r["id"] for r in hybrid_search(query, top_k=10)]
    results["dense"].append(recall_at_k(d_ids,  relevant, K))
    results["bm25" ].append(recall_at_k(b_ids,  relevant, K))
    results["hybrid"].append(recall_at_k(h_ids, relevant, K))
    mrrs["dense"].append(mrr(d_ids,  relevant))
    mrrs["bm25" ].append(mrr(b_ids,  relevant))
    mrrs["hybrid"].append(mrr(h_ids, relevant))

print(f"Eval over {len(LABELED)} queries, K={K}:\n")
print(f"{'Method':<10} {'Recall@3':>10} {'MRR':>10}")
for m in ["dense", "bm25", "hybrid"]:
    print(f"{m:<10} {np.mean(results[m]):>10.3f} {np.mean(mrrs[m]):>10.3f}")

# 💡 EXPERIMENT: try K=1 and K=5. Hybrid usually shines most at K=1 and K=3.
# 💡 EXPERIMENT: add adversarial queries with typos — BM25 falls off, dense stays up.


On real corpora, the typical pattern: **hybrid ≥ dense ≥ BM25** for most queries, with hybrid the most robust. The eval harness above is the same shape you'd build in CI to catch retrieval regressions when you change embedding models or chunking strategy.

**This connects directly to Lesson 17 (Advanced Evals)**: retrieval Recall@K is the *upstream* metric. Faithfulness/relevancy (RAGAS) are *downstream*. A drop in faithfulness is sometimes really a drop in retrieval recall — measure both.


---
## 8. Capstone — A pluggable `VectorStore` for your agent

Now we tie it together. The lesson is *not* "Pinecone is the right answer." The lesson is: **your agent code should not know which vector DB it's talking to**. Define an interface; swap implementations.

This is the same Java instinct you already have: `interface UserRepository { ... }` with `InMemoryUserRepository`, `PostgresUserRepository`, `MongoUserRepository`. Same pattern, vector edition.


In [ ]:
from abc import ABC, abstractmethod

@dataclass
class Document:
    id: str
    text: str
    metadata: Dict[str, Any] = field(default_factory=dict)
    embedding: Optional[np.ndarray] = None

@dataclass
class Hit:
    id: str
    score: float
    text: str
    metadata: Dict[str, Any]

class VectorStore(ABC):
    """Backend-agnostic vector-search contract."""

    @abstractmethod
    def upsert(self, docs: List[Document]) -> None: ...

    @abstractmethod
    def search(self, query_vec: np.ndarray, k: int = 5,
               filter: Optional[Dict[str, Any]] = None) -> List[Hit]: ...

    @abstractmethod
    def count(self) -> int: ...


# ---------- Impl 1: in-memory NumPy (great for tests) ----------
class NumpyStore(VectorStore):
    def __init__(self):
        self.docs: List[Document] = []
        self._matrix: Optional[np.ndarray] = None

    def upsert(self, docs):
        existing = {d.id: i for i, d in enumerate(self.docs)}
        for d in docs:
            if d.id in existing:
                self.docs[existing[d.id]] = d
            else:
                self.docs.append(d)
        self._matrix = np.stack([d.embedding for d in self.docs])

    def search(self, query_vec, k=5, filter=None):
        sims = self._matrix @ query_vec
        order = np.argsort(-sims)
        out: List[Hit] = []
        for i in order:
            d = self.docs[i]
            if filter and not _matches(d.metadata, filter):
                continue
            out.append(Hit(d.id, float(sims[i]), d.text, d.metadata))
            if len(out) >= k:
                break
        return out

    def count(self): return len(self.docs)


# ---------- Impl 2: sqlite-vec (pgvector-style) ----------
class SqliteVecStore(VectorStore):
    def __init__(self, dim: int):
        self.dim = dim
        self.con = sqlite3.connect(":memory:")
        self.con.enable_load_extension(True)
        sqlite_vec.load(self.con)
        self.con.enable_load_extension(False)
        self.con.executescript(f'''
            CREATE TABLE docs(id TEXT PRIMARY KEY, text TEXT, meta TEXT);
            CREATE VIRTUAL TABLE vecs USING vec0(embedding float[{dim}]);
        ''')
        self._id_to_rowid: Dict[str, int] = {}

    def upsert(self, docs):
        for d in docs:
            if d.id in self._id_to_rowid:
                rowid = self._id_to_rowid[d.id]
                self.con.execute("UPDATE docs SET text=?, meta=? WHERE id=?",
                                 (d.text, json.dumps(d.metadata), d.id))
                self.con.execute("DELETE FROM vecs WHERE rowid=?", (rowid,))
            else:
                cur = self.con.execute(
                    "INSERT INTO docs(id, text, meta) VALUES(?, ?, ?)",
                    (d.id, d.text, json.dumps(d.metadata)),
                )
                rowid = cur.lastrowid
                self._id_to_rowid[d.id] = rowid
            self.con.execute("INSERT INTO vecs(rowid, embedding) VALUES(?, ?)",
                             (rowid, d.embedding.tobytes()))
        self.con.commit()

    def search(self, query_vec, k=5, filter=None):
        # Over-fetch then post-filter (acceptable for small filters; pre-filter
        # for selective ones in real pgvector by adding WHERE to a joined query).
        rows = self.con.execute('''
            SELECT rowid, distance FROM vecs
             WHERE embedding MATCH ? ORDER BY distance LIMIT ?
        ''', (query_vec.tobytes(), k * 10)).fetchall()
        out: List[Hit] = []
        for rowid, dist in rows:
            r = self.con.execute("SELECT id, text, meta FROM docs WHERE rowid=?",
                                 (rowid,)).fetchone()
            if r is None: continue
            _id, text, meta_json = r
            meta = json.loads(meta_json)
            if filter and not _matches(meta, filter):
                continue
            # sqlite-vec 'distance' for cosine is 1 - cos_sim; convert to similarity.
            out.append(Hit(_id, 1.0 - float(dist), text, meta))
            if len(out) >= k:
                break
        return out

    def count(self):
        return self.con.execute("SELECT count(*) FROM docs").fetchone()[0]


# ---------- Impl 3: Pinecone (managed) ----------
class PineconeStore(VectorStore):
    def __init__(self, index_name: str, namespace: str = "default", dim: int = EMBED_DIM):
        from pinecone import Pinecone, ServerlessSpec
        if not PINECONE_API_KEY:
            raise RuntimeError("PINECONE_API_KEY missing")
        pc = Pinecone(api_key=PINECONE_API_KEY)
        if index_name not in [i.name for i in pc.list_indexes()]:
            pc.create_index(name=index_name, dimension=dim, metric="cosine",
                            spec=ServerlessSpec(cloud="aws", region="us-east-1"))
            while not pc.describe_index(index_name).status["ready"]:
                time.sleep(1)
        self.idx = pc.Index(index_name)
        self.ns  = namespace

    def upsert(self, docs):
        batch = [{"id": d.id, "values": d.embedding.tolist(),
                  "metadata": {**d.metadata, "_text": d.text}} for d in docs]
        self.idx.upsert(vectors=batch, namespace=self.ns)
        time.sleep(1)  # eventual consistency

    def search(self, query_vec, k=5, filter=None):
        res = self.idx.query(vector=query_vec.tolist(), top_k=k, namespace=self.ns,
                             filter=_to_pinecone_filter(filter),
                             include_metadata=True)
        out: List[Hit] = []
        for m in res["matches"]:
            meta = dict(m["metadata"])
            text = meta.pop("_text", "")
            out.append(Hit(m["id"], float(m["score"]), text, meta))
        return out

    def count(self):
        return self.idx.describe_index_stats().get("total_vector_count", 0)


# ---------- helpers ----------
def _matches(meta: Dict[str, Any], filter: Dict[str, Any]) -> bool:
    """Mongo-ish filter: {'tenant':'acme', 'year':{'$gte':2024}, 'topic':{'$in':['rag','agents']}}"""
    for key, cond in filter.items():
        val = meta.get(key)
        if isinstance(cond, dict):
            for op, target in cond.items():
                if op == "$gte" and not (val is not None and val >= target): return False
                if op == "$lte" and not (val is not None and val <= target): return False
                if op == "$eq"  and val != target: return False
                if op == "$in"  and val not in target: return False
        else:
            if val != cond: return False
    return True

def _to_pinecone_filter(f):
    return f  # already in Mongo-ish style, Pinecone accepts the same dialect

print("VectorStore interface + 3 implementations defined.")


In [ ]:
# Use the interface — flip a single line to change backends.

def build_documents() -> List[Document]:
    return [
        Document(id=str(d["id"]), text=d["body"],
                 metadata={"tenant": d["tenant"], "year": d["year"], "topic": d["topic"]},
                 embedding=embeddings[i])
        for i, d in enumerate(DOCS)
    ]

# 🔁 Swap this one line to change backends:
store: VectorStore = SqliteVecStore(dim=EMBED_DIM)
# store: VectorStore = NumpyStore()
# store: VectorStore = PineconeStore("learn-ai-lesson20", namespace="acme")  # if key set

store.upsert(build_documents())
print(f"Store: {type(store).__name__}   docs: {store.count()}")

q_vec = embed(["how do agents use tools?"])[0]
hits  = store.search(q_vec, k=3, filter={"tenant": "acme", "year": {"$gte": 2024}})

print("\nFiltered search results:")
for h in hits:
    print(f"  [{h.score:.3f}] ({h.metadata['topic']}/{h.metadata['year']}) {h.text[:80]}...")


In [ ]:
# Benchmark backends on the labeled eval set — same code, different backends.

def bench(store: VectorStore, label: str):
    store.upsert(build_documents())
    recalls, mrrs_, latencies = [], [], []
    for query, relevant in LABELED:
        q = embed([query])[0]
        t0 = time.time()
        hits = store.search(q, k=10)
        latencies.append((time.time() - t0) * 1000)
        ids = [int(h.id) for h in hits]
        recalls.append(recall_at_k(ids, relevant, K))
        mrrs_.append(mrr(ids, relevant))
    print(f"{label:<14}  Recall@{K}={np.mean(recalls):.3f}  MRR={np.mean(mrrs_):.3f}  "
          f"p50_latency={np.median(latencies):.1f}ms")

bench(NumpyStore(),               "NumpyStore")
bench(SqliteVecStore(EMBED_DIM),  "SqliteVecStore")
if PINECONE_API_KEY:
    bench(PineconeStore("learn-ai-lesson20-bench"), "PineconeStore")
else:
    print("PineconeStore   (skipped — no API key)")

# 💡 EXPERIMENT: extend the interface with `hybrid_search(query_text, ...)`
#                so each backend can implement RRF in its own optimal way
#                (Pinecone via sparse-dense, sqlite-vec via FTS5 + dense join).


**What you just built**: a vendor-agnostic vector layer your agent can sit on top of. When the team says "we're moving from Pinecone to self-hosted Qdrant to cut cost", you change one line of constructor and ship.

This is the *production* mindset that takes a RAG demo to a system: every external dependency hides behind an interface, every interface has a fake implementation for tests, every backend swap is a deploy-time decision, not a code-change.


---
## 9. Recap & what's next

You now have:

- A real intuition for **HNSW**: the dial is `ef`, the cost is memory, the win is sub-ms queries at 10M+ vectors.
- The **three production patterns**: pgvector/sqlite-vec (relational + vector), Pinecone (managed), self-hosted graph (Weaviate/Qdrant).
- **Hybrid search via RRF** — 8 lines of code that consistently beats dense-only.
- **MMR** for diverse top-K.
- A **labeled eval harness** for retrieval — the upstream half of the Lesson-17 RAGAS stack.
- A **pluggable VectorStore interface** with three implementations — the architecture pattern you'll keep using as you grow.

### Production checklist when you ship a vector-backed RAG system
1. ☐ Index choice deliberated (start: HNSW, M=16, ef_construction=200)
2. ☐ Recall@K measured against a labeled set, with a CI gate
3. ☐ Hybrid search wired up (you'll wish you had it the first time someone searches by error code)
4. ☐ Per-tenant isolation via namespaces or partial indexes
5. ☐ Embedding model + version recorded with every vector (reindex plan)
6. ☐ Filters use indexed metadata columns (pre-filter, not post-filter, when selective)
7. ☐ Observability: query latency p50/p99, recall regression alarms, per-tenant cost

### What's next (Lesson 21 preview)
**Agent Frameworks — CrewAI, AutoGen, comparison.** You've now built every layer of an AI agent system from scratch: tool-use, memory, multi-agent, RAG, structured outputs, evals, deployment, security, streaming, and vector infra. Lesson 21 surveys the **opinionated frameworks** that bundle these patterns — when to adopt one, when to keep building bespoke, and how the choices propagate into your AutoResearcher capstone.

> **Open-source angle:** The `VectorStore` + `hybrid_search` abstraction you wrote today is a clean, small library that would be useful on PyPI as a teaching/scaffolding tool — it sits between "low-level (Pinecone client)" and "framework (LangChain)" and is exactly the kind of focused, didactic project that gets stars. Worth keeping in mind as a candidate for your open-source play.

Happy building, Gourav. See you tomorrow.
